# Campus Access — YOLO + ByteTrack on Colab T4

Runs the entry/exit counter pipeline on Colab's free T4 GPU. Uses NVIDIA API key for LLM analysis of results.

In [1]:
# Install dependencies
!pip install -q ultralytics supervision opencv-python onnxruntime 2>&1 | tail -5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.8/865.8 kB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.3/144.3 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.8/75.8 kB 7.9 MB/s eta 0:00:00


In [2]:
# Check GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA available: False


In [10]:
# Download sample videos from GitHub using Git LFS
import os
import subprocess

repo_url = "https://github.com/Rawbeew/campus-access.git"
repo_dir = "campus-access"
videos = ["sample_input_pedestrian.mp4", "sample_input_vehicle.mp4"]

# Check if git-lfs is installed, install if not
try:
    subprocess.run(["git", "lfs", "version"], check=True, capture_output=True)
    print("Git LFS is installed.")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("Git LFS not found, installing...")
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "git-lfs"], check=True)
    subprocess.run(["git", "lfs", "install"], check=True)
    print("Git LFS installed.")

# Clone the repository if it doesn't exist
if not os.path.exists(repo_dir):
    print(f"Cloning {repo_url}...")
    # Use --depth=1 for a shallow clone to save time and space
    subprocess.run(["git", "clone", "--depth=1", repo_url], check=True)
    print("Repository cloned.")
else:
    print(f"Repository '{repo_dir}' already exists. Pulling latest LFS files...")
    current_dir = os.getcwd()
    os.chdir(repo_dir)
    # Pull LFS files in case they weren't downloaded or updated
    subprocess.run(["git", "lfs", "pull"], check=False) # check=False because pull might fail if no LFS files or not an LFS repo
    os.chdir(current_dir)

# Move the videos from the cloned directory to the current working directory
video_source_path = repo_dir # Corrected path: videos are directly in repo_dir
if os.path.exists(video_source_path):
    for v_name in videos:
        src_file = os.path.join(video_source_path, v_name)
        dst_file = v_name # Target filename in the current directory
        if os.path.exists(src_file) and not os.path.exists(dst_file):
            print(f"Moving {v_name} to current directory.")
            os.rename(src_file, dst_file)
        elif not os.path.exists(src_file):
            print(f"Warning: Source file {src_file} not found after cloning. LFS might have failed.")
        else:
            print(f"File {dst_file} already exists in current directory.")
else:
    print(f"Warning: Repository directory '{repo_dir}' not found. This should not happen after cloning.")

print("Video download/preparation complete.")

Git LFS is installed.
Repository 'campus-access' already exists. Pulling latest LFS files...
Moving sample_input_pedestrian.mp4 to current directory.
Moving sample_input_vehicle.mp4 to current directory.
Video download/preparation complete.


In [27]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv
import json
import time
import torch

# Load model
model = YOLO('yolov8n.pt')

# Set device based on CUDA availability
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Model moved to: {device}")

# Initialize trackers
ped_tracker = sv.ByteTrack()
veh_tracker = sv.ByteTrack()

# Line zones (same as local config)
ped_line = sv.LineZone(start=sv.Point(100, 100), end=sv.Point(500, 100))
veh_line = sv.LineZone(start=sv.Point(100, 200), end=sv.Point(500, 200))

# Annotators
box_annotator = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_thickness=1, text_scale=0.5)
line_annotator = sv.LineZoneAnnotator(thickness=2, text_thickness=1, text_scale=0.5)

cap = cv2.VideoCapture('sample_input_pedestrian.mp4')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_pedestrian_colab.mp4', fourcc, fps, (width, height))

ped_in = 0
ped_out = 0
frame_idx = 0
start = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # YOLO inference
    results = model(frame, verbose=False, conf=0.4)[0]
    detections = sv.Detections.from_ultralytics(results)

    # Filter for person class (0)
    detections = detections[detections.class_id == 0]

    # Track
    tracked = ped_tracker.update_with_detections(detections)

    # Line zone
    ped_line.trigger(tracked)

    # Annotate
    labels = [f"#{tid}" for tid in tracked.tracker_id] if tracked.tracker_id is not None else []
    annotated = box_annotator.annotate(scene=frame.copy(), detections=tracked)
    annotated = label_annotator.annotate(scene=annotated, detections=tracked, labels=labels)
    annotated = line_annotator.annotate(annotated, ped_line)

    out.write(annotated)
    frame_idx += 1

    if frame_idx % 50 == 0:
        print(f'  Frame {frame_idx}/{total_frames}')

cap.release()
out.release()

ped_in = ped_line.in_count
ped_out = ped_line.out_count
elapsed = time.time() - start

print(f'Pedestrian video done in {elapsed:.1f}s ({total_frames/elapsed:.1f} FPS)')
print(f'  IN: {ped_in}, OUT: {ped_out}')

ped_results = {'in': ped_in, 'out': ped_out, 'frames': frame_idx, 'time_s': elapsed}

Model moved to: cpu
  Frame 50/336
  Frame 100/336
  Frame 150/336
  Frame 200/336
  Frame 250/336
  Frame 300/336
Pedestrian video done in 33.2s (10.1 FPS)
  IN: 0, OUT: 0


In [14]:
# Run pipeline on vehicle video
veh_tracker = sv.ByteTrack()

# The model is already loaded and set to the correct device in the previous cell
# model.to('cuda') # No need to call model.to() again here

cap = cv2.VideoCapture('sample_input_vehicle.mp4')
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) # Corrected to CAP_PROP_FRAME_COUNT

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('output_vehicle_colab.mp4', fourcc, fps, (width, height))

veh_in = 0
veh_out = 0
frame_idx = 0
start = time.time()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, verbose=False, conf=0.4)[0]
    detections = sv.Detections.from_ultralytics(results)

    # Filter for vehicle classes (2=car, 3=motorcycle, 5=bus, 7=truck)
    vehicle_classes = [2, 3, 5, 7]
    mask = np.isin(detections.class_id, vehicle_classes)
    detections = detections[mask]

    tracked = veh_tracker.update_with_detections(detections)

    veh_line.trigger(tracked)

    labels = [f"#{tid}" for tid in tracked.tracker_id] if tracked.tracker_id is not None else []
    annotated = box_annotator.annotate(scene=frame.copy(), detections=tracked)
    annotated = label_annotator.annotate(scene=annotated, detections=tracked, labels=labels)
    annotated = line_annotator.annotate(annotated, veh_line)

    out.write(annotated)
    frame_idx += 1

    if frame_idx % 50 == 0:
        print(f'  Frame {frame_idx}/{total_frames}')

cap.release()
out.release()

veh_in = veh_line.in_count
veh_out = veh_line.out_count
elapsed = time.time() - start

print(f'Vehicle video done in {elapsed:.1f}s ({total_frames/elapsed:.1f} FPS)')
print(f'  IN: {veh_in}, OUT: {veh_out}')

veh_results = {'in': veh_in, 'out': veh_out, 'frames': frame_idx, 'time_s': elapsed}

  Frame 50/312
  Frame 100/312
  Frame 150/312
  Frame 200/312
  Frame 250/312
  Frame 300/312
Vehicle video done in 30.9s (10.1 FPS)
  IN: 0, OUT: 0


### Debugging Line Zone Coordinates

Since both pedestrian and vehicle counts are zero, let's visualize the line zones to ensure they are correctly placed within the video frames. The coordinates for `sv.Point(x, y)` define the line's start and end points relative to the video frame's dimensions (width, height).

In [30]:
import cv2
import matplotlib.pyplot as plt

# --- Visualize Pedestrian Line Zone ---
cap_ped = cv2.VideoCapture('sample_input_pedestrian.mp4')
ret_ped, frame_ped = cap_ped.read()
cap_ped.release()

if ret_ped:
    # Draw the pedestrian line zone on a sample frame
    frame_ped_copy = frame_ped.copy()
    cv2.line(frame_ped_copy, (ped_line.start.x, ped_line.start.y), (ped_line.end.x, ped_line.end.y), (0, 255, 0), 5)

    # Display the frame
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(frame_ped_copy, cv2.COLOR_BGR2RGB))
    plt.title('Pedestrian Video - Line Zone Visualization')
    plt.xlabel(f'Frame Width: {frame_ped.shape[1]}')
    plt.ylabel(f'Frame Height: {frame_ped.shape[0]}')
    plt.scatter([ped_line.start.x, ped_line.end.x], [ped_line.start.y, ped_line.end.y], color='red', s=100, label='Line Endpoints')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Could not read pedestrian video frame for visualization.")

AttributeError: 'LineZone' object has no attribute 'start'

In [31]:
import cv2
import matplotlib.pyplot as plt

# --- Visualize Vehicle Line Zone ---
cap_veh = cv2.VideoCapture('sample_input_vehicle.mp4')
ret_veh, frame_veh = cap_veh.read()
cap_veh.release()

if ret_veh:
    # Draw the vehicle line zone on a sample frame
    frame_veh_copy = frame_veh.copy()
    cv2.line(frame_veh_copy, (veh_line.start.x, veh_line.start.y), (veh_line.end.x, veh_line.end.y), (0, 0, 255), 5)

    # Display the frame
    plt.figure(figsize=(10, 6))
    plt.imshow(cv2.cvtColor(frame_veh_copy, cv2.COLOR_BGR2RGB))
    plt.title('Vehicle Video - Line Zone Visualization')
    plt.xlabel(f'Frame Width: {frame_veh.shape[1]}')
    plt.ylabel(f'Frame Height: {frame_veh.shape[0]}')
    plt.scatter([veh_line.start.x, veh_line.end.x], [veh_line.start.y, veh_line.end.y], color='red', s=100, label='Line Endpoints')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("Could not read vehicle video frame for visualization.")

AttributeError: 'LineZone' object has no attribute 'start'

### How to Adjust Line Zone Coordinates

After visualizing the line zones, if they are not correctly positioned to intersect with moving objects, you will need to adjust the `start` and `end` points. These are defined in cell `4jc8j9_fMPoD`:

```python
ped_line = sv.LineZone(start=sv.Point(100, 100), end=sv.Point(500, 100))
veh_line = sv.LineZone(start=sv.Point(100, 200), end=sv.Point(500, 200))
```

-   `sv.Point(x, y)`: `x` represents the horizontal position from the left edge of the frame, and `y` represents the vertical position from the top edge of the frame.
-   You can use the displayed frame dimensions (width and height on the plot axes) as a guide. For example, if a video is 1920 pixels wide and 1080 pixels tall, you would adjust `x` values between 0 and 1920, and `y` values between 0 and 1080.

**To make adjustments:**
1.  **Modify the `sv.Point` coordinates** in cell `4jc8j9_fMPoD`.
2.  **Rerun cell `4jc8j9_fMPoD`** to update the line zone objects.
3.  **Rerun the visualization cells** (the ones I just generated) to see the updated lines.
4.  Once the lines are correctly placed, **rerun the video processing cells** (`4jc8j9_fMPoD` and `egYl1cPHMPoE`) to get updated counts.

In [15]:
# Save results JSON
results = {
    "pedestrian": ped_results,
    "vehicle": veh_results,
    "summary": {
        "total_pedestrians": ped_results['in'] + ped_results['out'],
        "total_vehicles": veh_results['in'] + veh_results['out'],
        "ped_in": ped_results['in'],
        "ped_out": ped_results['out'],
        "veh_in": veh_results['in'],
        "veh_out": veh_results['out']
    }
}

with open('campus_access_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))

{
  "pedestrian": {
    "in": 0,
    "out": 0,
    "frames": 336,
    "time_s": 38.295461654663086
  },
  "vehicle": {
    "in": 0,
    "out": 0,
    "frames": 312,
    "time_s": 30.85053539276123
  },
  "summary": {
    "total_pedestrians": 0,
    "total_vehicles": 0,
    "ped_in": 0,
    "ped_out": 0,
    "veh_in": 0,
    "veh_out": 0
  }
}


In [29]:
import os
import json

# Install the Groq SDK
!pip install -q groq

# Import the Groq SDK
from groq import Groq
# Used to securely store your API key
from google.colab import userdata

# Get API key from Colab secrets
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

if not GROQ_API_KEY:
    print("GROQ_API_KEY not set in Colab secrets. Please set it via the '🔑' icon in the left panel.")
else:
    client = Groq(api_key=GROQ_API_KEY)

    # Using a suitable Groq model, e.g., 'llama3-8b-8192' (a currently supported model)
    groq_model_name = "llama3-8b-8192"
    print(f"Using Groq model: {groq_model_name}")

    prompt = f"""Analyze these campus entry/exit counting results:

Pedestrian gate: {ped_results['in']} entries, {ped_results['out']} exits
Vehicle gate: {veh_results['in']} entries, {veh_results['out']} exits

Provide:
1. Net flow (entries - exits) for each
2. Peak direction assessment
3. Any anomaly flags (e.g., more exits than entries = possible count error)
4. One-sentence operational summary for facilities team
"""

    try:
        chat_completion = client.chat.completions.create(
            messages=[
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model=groq_model_name,
        )
        analysis = chat_completion.choices[0].message.content

        print("=== LLM ANALYSIS ===")
        print(analysis)

        with open('llm_analysis.txt', 'w') as f:
            f.write(analysis)
    except Exception as e:
        print(f"Groq API error: {e}")
        print("Ensure your GROQ_API_KEY is valid and the model is accessible. Also check if the model is correctly specified.")

Using Groq model: llama3-8b-8192
Groq API error: Error code: 400 - {'error': {'message': 'The model `llama3-8b-8192` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
Ensure your GROQ_API_KEY is valid and the model is accessible. Also check if the model is correctly specified.


In [24]:
# Download outputs
from google.colab import files
files.download('output_pedestrian_colab.mp4')
files.download('output_vehicle_colab.mp4')
files.download('campus_access_results.json')
files.download('llm_analysis.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

FileNotFoundError: Cannot find file: llm_analysis.txt